In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_curve, roc_auc_score
import os

In [ ]:
my_data = pd.read_excel(r'C:\Users\dell\Documents\Customer Churn Analysis & Prediction\data_set\Telco_customer_churn.xlsx')
my_data.head()

In [ ]:
# Check column types 
my_data.info()

In [ ]:
# Quick statistics on numeric columns
my_data.describe()

In [ ]:
# Drop ID column

my_data.drop(columns='CustomerID', inplace = True) 

# Check the shape
print(f"Modeling dataset shape: {my_data.shape}")

In [ ]:
# Rename columns

new_columns = my_data.columns.str.replace(' ', '')
my_data.columns = new_columns

Exploratory Data Analysis


In [ ]:
# Count missing values
my_data.isnull().sum()

In [ ]:
# Check target variable
my_data['ChurnLabel'].value_counts()

In [ ]:
import matplotlib.pyplot as plt 
# Count values
counts = my_data['ChurnLabel'].value_counts()

# Labels (Yes / No)
labels = counts.index

# Plot pie chart
plt.pie(counts, labels=labels, autopct='%1.1f%%')

plt.title('Churn Distribution')
plt.show()

In [ ]:
import seaborn as sns
total = len(my_data)

ax = sns.countplot(x='Gender', hue='ChurnLabel', data=my_data)

# Annotate bars with percentages
for p in ax.patches:
    percentage = 100 * p.get_height() / total
    ax.annotate(f'{percentage:.1f}%', 
                (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom')

plt.title('Churn by Gender')
plt.xlabel('Gender')
plt.ylabel('Number of Customers')
plt.legend(title='Churn')
plt.show()

In [ ]:
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']


# Dictionary of comments for each column
comments = {
    'PhoneService': "Most customers have phone service.",
    'MultipleLines': "Customers without multiple lines have slightly lower churn.",
    'InternetService': "Fiber optic users tend to churn more; no internet service rarely churns.",
    'OnlineSecurity': "Customers without online security show higher churn.",
    'OnlineBackup': "Customers without online backup churn more; backup helps retention.",
    'DeviceProtection': "Lacking device protection correlates with higher churn.",
    'TechSupport': "No tech support is strongly linked to higher churn.",
    'StreamingTV': "Streaming TV may slightly decrease churn.",
    'StreamingMovies': "Similar to streaming TV, streaming movies have a small effect on churn."
}

# Loop through columns and plot
for col in service_cols:
    ax = sns.countplot(x=col, hue='ChurnLabel', data=my_data)
    plt.title(f"{col} vs Churn")
    
    # Show the plot
    plt.show()
    
    # Print the comment under the plot
    #print(f"Comment for {col}: {comments[col]}\n")
    print(f"{comments[col]}\n")

In [ ]:
 #Ensure numeric columns are truly numeric
numeric_cols = ['TenureMonths','MonthlyCharges','TotalCharges','CLTV']

for col in numeric_cols:
    my_data[col] = pd.to_numeric(my_data[col], errors='coerce')

# Optional: drop rows with missing values for these columns
my_data_clean = my_data.dropna(subset=numeric_cols)

numeric_comments = { 'TenureMonths': "Customers with shorter tenure are more likely to churn. Longer-tenured customers tend to stay, showing that loyalty increases over time.", 
                    'MonthlyCharges': "Customers with higher monthly charges show a slightly higher churn rate. Low-to-mid charges are associated with more retained customers.",
                    'TotalCharges': "Customers with low total charges tend to churn more, usually because they are newer customers. High total charges often indicate long-term customers who are less likely to churn.", 
                    'CLTV': "Customers with low CLTV are more likely to churn. Higher CLTV correlates with retention, suggesting these customers are more valuable and loyal." }

for col in numeric_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='ChurnLabel', y=col, data=my_data_clean)
    plt.title(f"{col} vs Churn")
    plt.xlabel("Churn")
    plt.ylabel(col)
    plt.show()
    
    print(f"{numeric_comments[col]}\n")

In [ ]:
# Correlation Heatmap (excluding geographic and leakage columns)
exclude_cols = ['Count','Country','State','City','ZipCode','LatLong','Latitude','Longitude','ChurnValue','ChurnScore']
heatmap_data = my_data.drop(columns=[c for c in exclude_cols if c in my_data.columns])

# Convert ChurnLabel to numeric for correlation
heatmap_data['ChurnLabel'] = heatmap_data['ChurnLabel'].map({'Yes': 1, 'No': 0})

plt.figure(figsize=(14, 10))
sns.heatmap(heatmap_data.corr(numeric_only=True), annot=False, cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Drop unimportant columns

# Drop geographic columns 
drop_cols = ['Count','Country','State','City','ZipCode','LatLong','Latitude','Longitude']
my_data.drop(columns=drop_cols, inplace = True)

# Drop ChurnValue and ChurnScore as they are derived from the target variable (ChurnLabel),
# Keeping them would cause data leakage and give the model the answer directly.

my_data.drop(columns = ['ChurnValue', 'ChurnScore'], inplace = True)

# Check the shape
print(my_data.shape)

In [ ]:
# Handle missing or inconsistent values

# 'Total Charges' and 'Monthly Charges'may have blank strings → convert to numeric; invalid parsing becomes NaN
cols_to_numeric = ['TotalCharges', 'MonthlyCharges']
my_data[cols_to_numeric] = my_data[cols_to_numeric].apply(pd.to_numeric, errors='coerce')

my_data.isnull().sum()

In [ ]:
# remove Churn Reason columns as it contains 73% of missing values
my_data.drop(columns = 'ChurnReason', inplace = True)

# Drop rows with missing Total Charges
my_data = my_data.dropna()

# Confirm no missing values
#print("Missing values per column after cleaning:")
print(my_data.isnull().sum())

In [ ]:
# Encode binary categorical variables

my_data = my_data.copy()

binary_cols = ['Gender','SeniorCitizen','Partner','Dependents','PhoneService','PaperlessBilling','ChurnLabel',
'MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']


for col in binary_cols:
    my_data[col] = my_data[col].replace({
        'Yes':1, 'No':0,
        'Male':1, 'Female':0,
        'No internet service':0,
        'No phone service':0
    })

# Quick check
my_data[binary_cols].head()

In [ ]:

#Scale numeric features

scaler = StandardScaler()
scale_cols = ['TenureMonths','MonthlyCharges','TotalCharges','CLTV']

my_data[scale_cols] = scaler.fit_transform(my_data[numeric_cols])

# Quick check
my_data[scale_cols].head()

In [ ]:
#One-hot encode multi-category columns

multi_cat_cols = ['InternetService','Contract','PaymentMethod']

my_data = pd.get_dummies(my_data, columns=multi_cat_cols, drop_first=True)

# Verify all columns are numeric
my_data.info()

In [ ]:
# Convert boolean columns to int
bool_cols = my_data.select_dtypes(include='bool').columns
my_data[bool_cols] = my_data[bool_cols].astype(int)

# Check
with pd.option_context('display.max_columns', None):  # None = show all columns
    display(my_data.head(5))

In [ ]:
my_data.head()

In [ ]:
# Define features and target

X = my_data.drop(['ChurnLabel'], axis=1)
y = my_data['ChurnLabel']

# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Train the model

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict

y_pred = model.predict(X_test)

# Evaluate

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


# Logistic Regression Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['No', 'Yes'], cmap='Blues')
plt.title('Logistic Regression - Confusion Matrix')
plt.show()

The model shows strong specificity (88.2%) but weak recall (60.7%), indicating a negative bias. It correctly identifies 911 non-cases but misses 147 positive cases. Accuracy is ~81%, but this masks class imbalance. Recommend adjusting decision threshold, using class weights, or applying SMOTE to improve recall on the minority class.

Random Forest

First, I will train a baseline Random Forest with default parameters to establish a benchmark .

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

# Predict

y_pred_rf = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

from sklearn.metrics import ConfusionMatrixDisplay

# Random Forest Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, display_labels=['No', 'Yes'], cmap='Blues')
plt.title('Random Forest - Confusion Matrix')
plt.show()

The baseline Random Forest shows similar accuracy to Logistic Regression, but with slightly lower churn recall.
Next, I will tune the hyperparameters to see if performance improves.

In [ ]:
# Tune Random Forest
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, 
                       cv=5, scoring='roc_auc', n_jobs=-1)
rf_grid.fit(X_train, y_train)

print("Best parameters:", rf_grid.best_params_)

# Use best model
rf = rf_grid.best_estimator_
y_pred_rf = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, display_labels=['No', 'Yes'], cmap='Blues')
plt.title('Random Forest (Tuned) - Confusion Matrix')
plt.show()

Tuning improved the model's overall AUC, but churn recall remains lower than Logistic Regression. 
The model favors precision over recall for the churn class

In [ ]:
# Top 16 most important features from Random Forest
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True).tail(16)

plt.figure(figsize=(8, 6))
feat_imp.plot(kind='barh')
plt.title('Top 16 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

The feature importance plot reveals which variables have the most influence on the model's predictions. TenureMonths, MonthlyCharges, TotalCharges, and Contract type rank among the top drivers, 
aligning with the patterns observed during EDA.

In [ ]:
# XGBoost
from xgboost import XGBClassifier

xgb = XGBClassifier(eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_xgb, display_labels=['No', 'Yes'], cmap='Blues')
plt.title('XGBoost - Confusion Matrix')
plt.show()

XGBoost shows competitive performance with Random Forest. Its strength lies in handling complex feature interactions, which helps capture subtle churn patterns that simpler models may miss.